In [14]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Install Streamlit and Ngrok
!pip install -q streamlit pyngrok

print("✅ Environment ready")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Environment ready


In [15]:
import os

# === Your source file path ===
source_path = "/content/drive/MyDrive/AIDM_7360_project/AIDM_7360_project.py"
# Target module name
target_path = "backend.py"

if not os.path.exists(source_path):
    print(f"❌ Error: File not found {source_path}")
else:
    # Read code
    with open(source_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    clean_lines = []
    for line in lines:
        # Auto-comment drive.mount as it causes errors in background
        if "drive.mount" in line:
            clean_lines.append(f"# [Auto-Commented] {line}")
        # Auto-comment main system entry (if you forgot to comment it out previously)
        elif "main_system_entry()" in line and "def" not in line:
            clean_lines.append(f"# [Auto-Commented] {line}")
        else:
            clean_lines.append(line)

    # Write new file
    with open(target_path, "w", encoding="utf-8") as f:
        f.writelines(clean_lines)

    print(f"✅ Success! Loaded core code '{source_path}'")
    print(f"   - Generated module: backend.py (Automatically sanitized drive.mount code)")

✅ Success! Loaded core code '/content/drive/MyDrive/AIDM_7360_project/AIDM_7360_project.py'
   - Generated module: backend.py (Automatically sanitized drive.mount code)


In [16]:
%%writefile app.py
import streamlit as st
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import sys
from io import StringIO
import os
import json

# Import backend logic
import backend as core_logic

# ==========================================
# 0. Setup & Configuration
# ==========================================
st.set_page_config(
    page_title="Steam Analytics Pro",
    layout="wide",
    page_icon="🎮",
    initial_sidebar_state="expanded"
)

DB_FILE_PATH = '/content/drive/MyDrive/AIDM_7360_project/games.db'

# Sync path to backend
core_logic.url = DB_FILE_PATH
core_logic.db_path = DB_FILE_PATH

# Report type mapping
REPORT_TYPES_MAP = {
    "developer_card": "🏭 Manufacturer's Card",
    "type_positioning": "🎯 Type Positioning",
    "recent_popularity": "📈 Recent Popularity",
    "multi_platform_fit": "💻 Platform Fit",
    "value_king": "💰 Value King",
    "comprehensive_recommendation": "🏆 Comprehensive Recommendation"
}

@st.cache_resource
def get_conn():
    if os.path.exists(DB_FILE_PATH):
        return sqlite3.connect(DB_FILE_PATH, check_same_thread=False)
    return None

conn = get_conn()

# ==========================================
# 1. Shared UI Components & Helper Functions
# ==========================================

def colored_number_html(value, thresholds=(1000000, 100000)):
    """Returns HTML string with colored number based on magnitude."""
    try:
        v = int(value)
    except Exception:
        return f"<span>{value}</span>"
    if v >= thresholds[0]:
        color = '#10b981'  # Green
    elif v >= thresholds[1]:
        color = '#f59e0b'  # Amber
    else:
        color = '#ef4444'  # Red
    return f"<span style='color:{color};font-weight:700;'>{v:,}</span>"

def percent_color_html(pct):
    """Returns HTML string with colored percentage."""
    try:
        p = float(pct)
    except Exception:
        return f"<span>{pct}</span>"
    if p >= 90:
        color = '#10b981'
    elif p >= 70:
        color = '#f59e0b'
    else:
        color = '#ef4444'
    return f"<span style='color:{color};font-weight:700;'>{p:.0f}%</span>"

def render_game_card(row):
    """Renders a modern game card with image and key metrics."""
    header = row.get('Header_image', '')
    name = row.get('Name', 'Unknown')
    price = row.get('Price', 0)
    pos_rate = row.get('PositiveRate', 0)
    metacritic = row.get('Metacritic_score', 'N/A')
    positive = row.get('Positive', 0)

    price_text = f"${price:.2f}" if price and price > 0 else '🆓 Free'
    pos_html = percent_color_html(pos_rate)

    # Default placeholder image
    placeholder_url = "https://via.placeholder.com/300x150/1e293b/94a3b8?text=No+Image"

    card_html = f"""
    <div style="display:flex; padding:15px; border-radius:12px; margin-bottom:15px; background-color:#1e293b; color:#e2e8f0; border:1px solid #334155; box-shadow:0 4px 6px -1px rgba(0, 0, 0, 0.1); transition:transform 0.2s;">
        <div style="flex:0 0 200px; display:flex; align-items:center;">
            <img src="{header or placeholder_url}" style="width:100%; height:120px; border-radius:8px; object-fit:cover;" onerror="this.src='{placeholder_url}'"/>
        </div>
        <div style="flex:1; padding-left:20px; display:flex; flex-direction:column; justify-content:center;">
            <h3 style="margin:0 0 10px 0; color:#f8fafc; font-size:20px; line-height:1.2;">{name}</h3>
            <div style="font-size:14px; color:#94a3b8; margin-bottom:8px;">
                🏷️ Price: <span style="color:#fff; font-weight:bold; margin-right:20px;">{price_text}</span>
                👍 Rating: {pos_html}
            </div>
            <div style="font-size:12px; color:#64748b;">
                🎯 Metacritic: {metacritic} | 📊 Positive Reviews: {positive:,}
            </div>
        </div>
    </div>
    """
    st.markdown(card_html, unsafe_allow_html=True)

def get_game_extended_data(game_name, db_path):
    """Get extended game data including tags, screenshots, etc."""
    try:
        conn_temp = sqlite3.connect(db_path)
        extended_sql = """
        SELECT
            g.*,
            GROUP_CONCAT(DISTINCT d.Name) as Developers,
            GROUP_CONCAT(DISTINCT p.Name) as Publishers,
            GROUP_CONCAT(DISTINCT gn.Name) as Genres,
            GROUP_CONCAT(DISTINCT t.Name) as Tags,
            g.Screenshots as Screenshots
        FROM Game g
        LEFT JOIN Game_Developer gd ON g.AppID = gd.AppID
        LEFT JOIN Developer d ON gd.DeveloperID = d.DeveloperID
        LEFT JOIN Game_Publisher gp ON g.AppID = gp.AppID
        LEFT JOIN Publisher p ON gp.PublisherID = p.PublisherID
        LEFT JOIN Game_Genre gg ON g.AppID = gg.AppID
        LEFT JOIN Genre gn ON gg.GenreID = gn.GenreID
        LEFT JOIN Game_Tag gt ON g.AppID = gt.AppID
        LEFT JOIN Tag t ON gt.TagID = t.TagID
        WHERE g.Name = ?
        GROUP BY g.AppID
        """
        game_df = pd.read_sql_query(extended_sql, conn_temp, params=(game_name,))
        conn_temp.close()

        if not game_df.empty:
            return game_df.iloc[0].to_dict()
        return None
    except Exception as e:
        st.error(f"Error getting extended data: {e}")
        return None

def parse_screenshots(screenshots_data):
    """Parse screenshot data."""
    if not screenshots_data:
        return []
    try:
        if isinstance(screenshots_data, str):
            if screenshots_data.startswith('['):
                return json.loads(screenshots_data)
            else:
                return [url.strip() for url in screenshots_data.split(',') if url.strip()]
        elif isinstance(screenshots_data, list):
            return screenshots_data
    except:
        pass
    return []

def render_modern_report_header(game_data):
    """Renders the top visual section of the report with enhanced content."""
    if not game_data:
        return

    # 1. Title & Hero Image
    st.markdown(f"# 🎮 {game_data.get('Name', 'Unknown Game')}")

    if game_data.get('Header_image'):
        st.image(game_data['Header_image'], use_container_width=True, caption=f"{game_data.get('Name')} - Header Image")

    # 2. KPI Metrics
    positive = game_data.get('Positive', 0) or 0
    negative = game_data.get('Negative', 0) or 0
    total = positive + negative
    pos_rate = (positive / total * 100) if total > 0 else 0
    peak_ccu = game_data.get('Peak_ccu', 0) or 0
    price = game_data.get('Price', 0) or 0

    k1, k2, k3, k4 = st.columns(4)
    with k1:
        st.metric("👍 Positive", f"{positive:,}", delta=f"{pos_rate:.1f}%")
    with k2:
        st.metric("👎 Negative", f"{negative:,}")
    with k3:
        st.metric("🔥 Peak CCU", f"{peak_ccu:,}")
    with k4:
        price_text = "Free" if price == 0 else f"${price:.2f}"
        st.metric("💰 Price", price_text)

    st.markdown("---")

    # 3. HTML Stats Bar
    avg_play = game_data.get('Average_playtime_forever', 0) or 0
    recommendations = game_data.get('Recommendations', 0) or 0
    achievements = game_data.get('Achievements', 0) or 0

    stats_html = f"""
    <div style='display:flex; gap:15px; flex-wrap:wrap; margin-bottom:20px;'>
        <div style='background:#0f172a; padding:15px; border-radius:8px; border-left:4px solid #3b82f6; flex:1; min-width:120px;'>
            <span style='color:#94a3b8; font-size:12px;'>⏱️ Avg Playtime</span><br>
            <strong style='color:#e2e8f0; font-size:16px;'>{avg_play//60}h {avg_play%60}m</strong>
        </div>
        <div style='background:#0f172a; padding:15px; border-radius:8px; border-left:4px solid #10b981; flex:1; min-width:120px;'>
            <span style='color:#94a3b8; font-size:12px;'>📈 Total Recs</span><br>
            <span style='font-size:16px;'>{colored_number_html(recommendations)}</span>
        </div>
        <div style='background:#0f172a; padding:15px; border-radius:8px; border-left:4px solid #f59e0b; flex:1; min-width:120px;'>
            <span style='color:#94a3b8; font-size:12px;'>🏆 Metacritic</span><br>
            <strong style='color:#e2e8f0; font-size:16px;'>{game_data.get('Metacritic_score', 'N/A')}</strong>
        </div>
        <div style='background:#0f172a; padding:15px; border-radius:8px; border-left:4px solid #8b5cf6; flex:1; min-width:120px;'>
            <span style='color:#94a3b8; font-size:12px;'>🎯 Achievements</span><br>
            <strong style='color:#e2e8f0; font-size:16px;'>{achievements}</strong>
        </div>
    </div>
    """
    st.markdown(stats_html, unsafe_allow_html=True)

    # 4. Basic Info & Tags
    col_info, col_tags = st.columns(2)

    with col_info:
        st.subheader("📋 Basic Information")
        developers = game_data.get('Developers', 'Unknown')
        publishers = game_data.get('Publishers', 'Unknown')
        genres = game_data.get('Genres', 'Unknown')
        release_date = game_data.get('Release_date', 'Unknown')

        st.markdown(f"""
        **Developers:** {developers}
        **Publishers:** {publishers}
        **Genres:** {genres}
        **Release Date:** {release_date}
        **Supported OS:** {'Windows' if game_data.get('Windows') else ''} {'Mac' if game_data.get('Mac') else ''} {'Linux' if game_data.get('Linux') else ''}
        """)

    with col_tags:
        st.subheader("🏷️ Popular Tags")
        tags = game_data.get('Tags', '')
        if tags:
            tag_list = [tag.strip() for tag in tags.split(',') if tag.strip()]
            tag_html = " ".join([f"<span style='background:#334155; color:#e2e8f0; padding:4px 8px; border-radius:12px; font-size:12px; margin:2px; display:inline-block;'>{tag}</span>" for tag in tag_list[:10]])
            st.markdown(tag_html, unsafe_allow_html=True)
        else:
            st.info("No tags available")

    # 5. Description
    about_text = game_data.get('About_the_game') or game_data.get('detailed_description') or game_data.get('Short_description') or "No description available."
    with st.expander("📖 Game Description", expanded=True):
        st.write(about_text)

    # 6. Screenshot Gallery
    screenshots = parse_screenshots(game_data.get('Screenshots'))
    if screenshots:
        st.markdown("---")
        st.subheader("📸 Screenshot Gallery")
        cols = st.columns(3)
        for i, screenshot_url in enumerate(screenshots[:6]):
            with cols[i % 3]:
                st.image(screenshot_url, use_container_width=True, caption=f"Screenshot {i+1}")

    st.markdown("---")

def render_full_report(game_name):
    """Generates and displays the full deep dive report."""
    with st.spinner("🔍 Gathering game data and generating analysis..."):
        # Get extended game data (including tags, screenshots, etc.)
        game_data = get_game_extended_data(game_name, DB_FILE_PATH)
        all_templates = core_logic.get_all_templates(DB_FILE_PATH)

        if not game_data:
            st.error(f"❌ Game '{game_name}' not found in database.")
            return

        if not all_templates:
            st.error("❌ Could not load report templates.")
            return

        # === Enhanced Modern Report Header ===
        render_modern_report_header(game_data)

        # === AI Analysis Sections ===
        st.header("🧠 Detailed Analysis")

        # Define report sections with icons
        report_sections = [
            ('comprehensive_recommendation', '🏆 Final Verdict'),
            ('value_king', '💰 Value Analysis'),
            ('type_positioning', '🎯 Type Positioning'),
            ('developer_card', '🏭 Developer Profile'),
            ('recent_popularity', '📈 Recent Popularity'),
            ('multi_platform_fit', '💻 Platform Compatibility')
        ]

        # Create two columns for balanced layout
        col1, col2 = st.columns(2)

        for i, (report_key, title) in enumerate(report_sections):
            if report_key in all_templates:
                # Get the generator function from backend
                func_name = f"generate_{report_key}"
                if hasattr(core_logic, func_name):
                    generator_func = getattr(core_logic, func_name)

                    # Generate the report content
                    try:
                        report_content = generator_func(game_data, all_templates[report_key])
                        # Clean up the content (remove header lines)
                        content_lines = report_content.split('\n')
                        clean_content = '\n'.join(content_lines[1:]) if len(content_lines) > 1 else report_content

                        # Display in expander
                        target_col = col1 if i % 2 == 0 else col2
                        with target_col:
                            with st.expander(title, expanded=True):
                                st.markdown(clean_content)
                    except Exception as e:
                        st.error(f"Error generating {title}: {e}")

        st.success("✅ Analysis Report Generated Successfully!")

# ==========================================
# 2. Options Data & Filter Mapping
# ==========================================
OPT = {
    'q1': {
        1: 'Action', 2: 'Adventure', 3: 'Casual', 4: 'RPG', 5: 'Strategy', 6: 'Simulation',
        7: 'Indie', 8: 'Sports', 9: 'Racing', 10: 'Massively Multiplayer', 11: 'Free to Play',
        12: 'Early Access', 13: 'Mature ', 14: 'Episodic', 15: 'Education ', 16: 'Modeling',
        17: 'Video Production', 18: 'Audio Production', 19: 'Game Dev & Web', 20: 'Utilities'
    },
    'q2_1': {1: 'Pixel', 2: '8-bit', 3: '80s/90s', 4: 'Hand-drawn', 5: 'Cartoon', 6: 'Comic', 7: 'Anime',
             8: 'Realistic', 9: 'Stylized', 10: 'Voxel', 11: 'Abstract', 12: 'Minimalist', 13: 'Cinematic'},
    'q2_2': {1: 'First-Person', 2: 'Third Person', 3: 'Top-Down', 4: 'Side Scroller'},
    'q3_1': {1: 'Cyberpunk', 2: 'Space', 3: 'Post-apoc', 4: 'Robots', 5: 'Fantasy', 6: 'Mythology',
             7: 'Dark Fantasy', 8: 'Steampunk', 9: 'Modern', 10: 'History/War', 11: 'Military', 12: 'Crime',
             13: 'Lovecraft', 14: 'Zombies', 15: 'Vampire', 16: 'Dystopian'},
    'q3_2': {1: 'Story Rich', 2: 'Visual Novel', 3: 'Choices Matter', 4: 'Horror/Mystery', 5: 'Comedy', 6: 'Drama',
             7: 'Philosophical'},
    'q4_1': {1: 'Casual/Cute', 2: 'Action/Thriller', 3: 'Horror', 4: 'Cozy', 5: 'Dark/Gore', 6: 'Epic',
             7: 'Atmospheric'},
    'q4_2': {1: 'Casual', 2: 'Difficult', 3: 'Souls-like', 6: 'Adjustable'},
    'q5_1': {1: 'Local Co-Op', 2: 'Online Co-Op', 3: 'PvP', 4: 'Team-Based', 5: 'Battle Royale', 6: 'MMO',
             7: 'Social Deduction', 8: 'Singleplayer'},
    'q6_1': {1: 'Moddable', 2: 'Replay Value', 3: 'Great Soundtrack'},
    'q6_2': {1: 'VR', 2: 'Controller'},
    'q6_3': {1: 'Family Friendly', 2: 'Mature/Nudity', 3: 'Violent/Gore', 4: 'Horror'},
    'q7_1': {1: 'Free to Play', 2: '<10$', 3: '10-30$', 4: '30-60$', 5: '>60$'},
    'q7_2': {1: 'No DLC', 2: 'Small amount', 3: 'Lots of DLC'},
    'q8_1': {1: 'Rave reviews(95%+)', 2: 'Very Positive(80%+)', 3: 'Mostly Positive(70%+)', 4: 'Mixed(40%+)'},
    'q9_1': {1: '<10h', 2: '10-30h', 3: '30-100h', 4: '>100h'},
    'q9_2': {1: 'Achievements(>20)', 2: 'Multiple Endings', 3: 'Procedural Generation', 4: 'Workshop', 5: 'Multiplayer elements'},
    'q10_1': {1: 'Win Only', 2: 'Mac Only', 3: 'Linux Only', 4: 'Cross-platform'},
    'q10_2': {1: 'Simplified Chinese', 2: 'English', 3: 'Japanese'},
    'qf_1': {1: '2024', 2: '2022-2024', 3: '<2022'}
}

LOGIC_GROUPS = {
    'need_perspective': [1, 2, 4, 6, 8, 9, 10],
    'need_story': [2, 4, 14],
    'hide_difficulty': [3, 6, 8, 14, 15, 16, 17, 18, 19, 20],
    'need_multiplayer': [10],
    'need_hardware': [6, 9]
}

FILTER_MAP = {
    'q2_1': core_logic.Q2_1_Style_SQL, 'q2_2': core_logic.Q2_2_Perspective_SQL,
    'q3_1': core_logic.Q3_1_Theme_SQL, 'q3_2': core_logic.Q3_2_Story_SQL, 'q6_3': core_logic.Q6_3_Content_SQL,
    'q4_1': core_logic.Q4_1_Mood_SQL, 'q4_2': core_logic.Q4_2_Difficulty_SQL,
    'q5_1': core_logic.Q5_1_Multiplayer_SQL,
    'q6_1': core_logic.Q6_1_Features_SQL, 'q6_2': core_logic.Q6_2_Hardware_SQL,
    'q7_1': core_logic.Q7_1_Price_SQL, 'q7_2': core_logic.Q7_2_DLC_SQL,
    'q8_1': core_logic.Q8_1_Review_SQL,
    'q9_1': core_logic.Q9_1_Playtime_SQL, 'q9_2': core_logic.Q9_2_Replay_SQL,
    'q10_1': core_logic.Q10_1_OS_SQL, 'q10_2': core_logic.Q10_2_Lang_SQL,
    'qf_1': core_logic.QF_1_Year_SQL
}

# ==========================================
# 3. Main Application Logic
# ==========================================
def main():
    if conn is None:
        st.error("❌ Database connection failed. Please check the database file path.")
        return

    st.sidebar.title("🎮 Steam Analytics Pro")
    st.sidebar.markdown("---")

    mode = st.sidebar.radio(
        "Navigation",
        ["🔍 Filter Dashboard", "📑 Deep Dive Report", "🛠️ Database Admin"],
        index=0
    )

    # -------------------------------------
    # MODE 1: Modern Filter Dashboard
    # -------------------------------------
    if mode == "🔍 Filter Dashboard":
        st.title("🔍 Interactive Game Finder")
        st.caption("Filter games and view them as modern cards with images.")

        if 'filter_results_df' not in st.session_state:
            st.session_state['filter_results_df'] = None

        if 'dashboard_report_game' not in st.session_state:
            st.session_state['dashboard_report_game'] = None

        # 1. Core Genre Selection
        st.subheader("1. Select Core Genre")
        genre_options = {0: "🎯 Select Genre..."}
        genre_options.update(OPT['q1'])
        genre_key = st.selectbox("Genre", options=list(genre_options.keys()),
                               format_func=lambda x: genre_options[x])

        active_filters = {}
        if genre_key != 0:
            active_filters['q1'] = core_logic.Q1_Genre_SQL(genre_key)

            # 2. Complete Filter System
            st.subheader("2. Refine Your Search")

            # Filter Layout
            col1, col2 = st.columns(2)

            with col1:
                # 🎨 Visuals & Presentation
                with st.expander("🎨 Visuals & Presentation", expanded=True):
                    style_val = st.selectbox("Art Style", [0] + list(OPT['q2_1'].keys()),
                                           format_func=lambda x: "Any" if x == 0 else OPT['q2_1'][x])
                    if style_val != 0:
                        active_filters['q2_1'] = FILTER_MAP['q2_1'](style_val)

                    # Logic: Perspective
                    if genre_key in LOGIC_GROUPS['need_perspective']:
                        perspective_val = st.selectbox("Perspective", [0] + list(OPT['q2_2'].keys()),
                                                     format_func=lambda x: "Any" if x == 0 else OPT['q2_2'][x])
                        if perspective_val != 0:
                            active_filters['q2_2'] = FILTER_MAP['q2_2'](perspective_val)

                # 🎭 Content & Theme
                with st.expander("🎭 Content & Theme"):
                    theme_val = st.selectbox("World Setting", [0] + list(OPT['q3_1'].keys()),
                                           format_func=lambda x: "Any" if x == 0 else OPT['q3_1'][x])
                    if theme_val != 0:
                        active_filters['q3_1'] = FILTER_MAP['q3_1'](theme_val)

                    content_val = st.selectbox("Content Rating", [0] + list(OPT['q6_3'].keys()),
                                             format_func=lambda x: "Any" if x == 0 else OPT['q6_3'][x])
                    if content_val != 0:
                        active_filters['q6_3'] = FILTER_MAP['q6_3'](content_val)

                    # Logic: Storytelling
                    if genre_key in LOGIC_GROUPS['need_story']:
                        story_val = st.selectbox("Storytelling", [0] + list(OPT['q3_2'].keys()),
                                               format_func=lambda x: "Any" if x == 0 else OPT['q3_2'][x])
                        if story_val != 0:
                            active_filters['q3_2'] = FILTER_MAP['q3_2'](story_val)

                # ⚙️ Tech & Features
                with st.expander("⚙️ Tech & Features"):
                    features_val = st.selectbox("Features", [0] + list(OPT['q6_1'].keys()),
                                              format_func=lambda x: "Any" if x == 0 else OPT['q6_1'][x])
                    if features_val != 0:
                        active_filters['q6_1'] = FILTER_MAP['q6_1'](features_val)

                    # Logic: Hardware
                    if genre_key in LOGIC_GROUPS['need_hardware']:
                        hardware_val = st.selectbox("Hardware", [0] + list(OPT['q6_2'].keys()),
                                                  format_func=lambda x: "Any" if x == 0 else OPT['q6_2'][x])
                        if hardware_val != 0:
                            active_filters['q6_2'] = FILTER_MAP['q6_2'](hardware_val)

            with col2:
                # 🕹️ Experience
                with st.expander("🕹️ Experience"):
                    mood_val = st.selectbox("Atmosphere", [0] + list(OPT['q4_1'].keys()),
                                          format_func=lambda x: "Any" if x == 0 else OPT['q4_1'][x])
                    if mood_val != 0:
                        active_filters['q4_1'] = FILTER_MAP['q4_1'](mood_val)

                    # Logic: Difficulty
                    if genre_key not in LOGIC_GROUPS['hide_difficulty']:
                        difficulty_val = st.selectbox("Difficulty", [0] + list(OPT['q4_2'].keys()),
                                                    format_func=lambda x: "Any" if x == 0 else OPT['q4_2'][x])
                        if difficulty_val != 0:
                            active_filters['q4_2'] = FILTER_MAP['q4_2'](difficulty_val)

                # 👥 Multiplayer
                with st.expander("👥 Multiplayer"):
                    multiplayer_val = st.selectbox("Multiplayer Mode", [0] + list(OPT['q5_1'].keys()),
                                                 format_func=lambda x: "Any" if x == 0 else OPT['q5_1'][x])
                    if multiplayer_val != 0:
                        active_filters['q5_1'] = FILTER_MAP['q5_1'](multiplayer_val)

                # 💰 Price & Value
                with st.expander("💰 Price & Value"):
                    price_val = st.selectbox("Price Range", [0] + list(OPT['q7_1'].keys()),
                                           format_func=lambda x: "Any" if x == 0 else OPT['q7_1'][x])
                    if price_val != 0:
                        active_filters['q7_1'] = FILTER_MAP['q7_1'](price_val)

                    dlc_val = st.selectbox("DLC Preference", [0] + list(OPT['q7_2'].keys()),
                                         format_func=lambda x: "Any" if x == 0 else OPT['q7_2'][x])
                    if dlc_val != 0:
                        active_filters['q7_2'] = FILTER_MAP['q7_2'](dlc_val)

                    review_val = st.selectbox("Review Score", [0] + list(OPT['q8_1'].keys()),
                                            format_func=lambda x: "Any" if x == 0 else OPT['q8_1'][x])
                    if review_val != 0:
                        active_filters['q8_1'] = FILTER_MAP['q8_1'](review_val)

                    playtime_val = st.selectbox("Playtime", [0] + list(OPT['q9_1'].keys()),
                                              format_func=lambda x: "Any" if x == 0 else OPT['q9_1'][x])
                    if playtime_val != 0:
                        active_filters['q9_1'] = FILTER_MAP['q9_1'](playtime_val)

                # 🖥️ Compatibility
                with st.expander("🖥️ Compatibility"):
                    os_val = st.selectbox("Operating System", [0] + list(OPT['q10_1'].keys()),
                                        format_func=lambda x: "Any" if x == 0 else OPT['q10_1'][x])
                    if os_val != 0:
                        active_filters['q10_1'] = FILTER_MAP['q10_1'](os_val)

                    lang_val = st.selectbox("Language", [0] + list(OPT['q10_2'].keys()),
                                          format_func=lambda x: "Any" if x == 0 else OPT['q10_2'][x])
                    if lang_val != 0:
                        active_filters['q10_2'] = FILTER_MAP['q10_2'](lang_val)

            # 📅 Release Year & Sort Options
            st.subheader("📅 Release Time & Sorting")
            year_col1, year_col2 = st.columns([1, 1])

            with year_col1:
                year_val = st.selectbox("Release Year", [0] + list(OPT['qf_1'].keys()),
                                      format_func=lambda x: "Any" if x == 0 else OPT['qf_1'][x])
                if year_val != 0:
                    active_filters['qf_1'] = FILTER_MAP['qf_1'](year_val)

            with year_col2:
                sort_option = st.selectbox(
                    "Sort Results By",
                    ["Most Popular (Reviews)", "Highest Rated (Metacritic)", "Lowest Price", "Highest Price", "Most Playtime"],
                    index=0
                )

            # 3. Apply Filters Button
            st.markdown("---")
            if st.button("🚀 Apply Filters", type="primary", use_container_width=True):
                # Reset report view
                st.session_state['dashboard_report_game'] = None

                with st.spinner("Searching games..."):
                    result_ids = core_logic.execute_intersect(conn, active_filters)

                    if not result_ids:
                        st.warning("No games found matching your criteria.")
                        st.session_state['filter_results_df'] = None
                    else:
                        st.success(f"🎉 Found {len(result_ids)} games!")
                        ids = tuple(result_ids)
                        if len(ids) == 1:
                            ids = f"({ids[0]})"

                        # Determine sorting
                        sort_map = {
                            "Most Popular (Reviews)": "g.Positive DESC",
                            "Highest Rated (Metacritic)": "g.Metacritic_score DESC",
                            "Lowest Price": "g.Price ASC",
                            "Highest Price": "g.Price DESC",
                            "Most Playtime": "g.Average_playtime_forever DESC"
                        }
                        order_clause = sort_map.get(sort_option, "g.Positive DESC")

                        # Enhanced SQL
                        data_sql = f"""
                        SELECT g.AppID, g.Name, g.Price, g.Metacritic_score, g.Positive, g.Negative,
                               g.Header_image, g.Average_playtime_forever, g.Peak_ccu, g.Achievements
                        FROM Game g WHERE g.AppID IN {ids}
                        ORDER BY {order_clause} LIMIT 50
                        """
                        try:
                            df = pd.read_sql_query(data_sql, conn)
                            df['PositiveRate'] = df.apply(
                                lambda x: int((x['Positive']/(x['Positive']+x['Negative']))*100)
                                if (x['Positive']+x['Negative']) > 0 else 0, axis=1
                            )
                            st.session_state['filter_results_df'] = df
                        except Exception as e:
                            st.error(f"SQL Error: {e}")

            # 4. Modern Results Display
            if st.session_state['filter_results_df'] is not None:
                df = st.session_state['filter_results_df']
                st.divider()
                st.subheader(f"📋 Results ({len(df)} games)")

                # Display games as modern cards
                for idx, row in df.iterrows():
                    render_game_card(row)

                # 5. Analysis Bridge (Directly in Dashboard)
                st.divider()
                st.subheader("📊 Want deeper analysis?")
                analysis_col1, analysis_col2 = st.columns([3, 1])

                with analysis_col1:
                    target_game = st.selectbox("Select a game for detailed analysis:",
                                             df['Name'].unique(), key="dashboard_analysis_select")

                with analysis_col2:
                    if st.button("Deep Dive 🎯", type="secondary", use_container_width=True, key="dashboard_deep_dive_btn"):
                        st.session_state['dashboard_report_game'] = target_game
                        st.rerun()

        # 6. Render Report at Bottom of Dashboard if selected
        if st.session_state.get('dashboard_report_game'):
            st.divider()
            st.markdown(f"## 🔎 Report for: {st.session_state['dashboard_report_game']}")

            if st.button("❌ Close Report", key="close_report"):
                st.session_state['dashboard_report_game'] = None
                st.rerun()

            render_full_report(st.session_state['dashboard_report_game'])


    # -------------------------------------
    # MODE 2: Enhanced Deep Dive Report
    # -------------------------------------
    elif mode == "📑 Deep Dive Report":
        st.title("📑 Game Analysis Report")
        st.caption("Enter a game name to search directly.")

        # Pre-fill if coming from legacy route
        default_name = ""
        game_name = st.text_input("Enter exact game name:", value=default_name,
                                placeholder="e.g., Black Myth: Wukong")

        if st.button("Generate Report 🚀", type="primary") and game_name:
             render_full_report(game_name)

    # -------------------------------------
    # MODE 3: Database Admin
    # -------------------------------------
    elif mode == "🛠️ Database Admin":
        st.title("🛠️ Database Management")
        st.warning("⚠️ Operations here directly modify the database.")

        tab1, tab2, tab3 = st.tabs(["🆕 Add Game", "✏️ Update Price", "🗑️ Delete Game"])
        cursor = conn.cursor()

        with tab1:
            with st.form("add_game"):
                st.subheader("Add New Game")
                name = st.text_input("Game Name")
                price = st.number_input("Price ($)", min_value=0.0, step=0.01)
                submitted = st.form_submit_button("Add Game")
                if submitted and name:
                    try:
                        cursor.execute("SELECT MAX(AppID) FROM Game")
                        max_id = cursor.fetchone()[0]
                        new_id = (max_id + 1) if max_id else 999900
                        cursor.execute(
                            "INSERT INTO Game (AppID, Name, Price, Release_date, Positive, Negative, User_score) VALUES (?, ?, ?, 'Jan 1, 2025', 0, 0, 0)",
                            (new_id, name, price)
                        )
                        conn.commit()
                        st.success(f"✅ Added '{name}' with ID {new_id}")
                    except Exception as e:
                        st.error(f"Error: {e}")

        with tab2:
            with st.form("update_price"):
                st.subheader("Update Price")
                u_name = st.text_input("Game Name to Update")
                u_price = st.number_input("New Price ($)", min_value=0.0, step=0.01)
                submitted = st.form_submit_button("Update")
                if submitted and u_name:
                    try:
                        cursor.execute("UPDATE Game SET Price = ? WHERE Name = ?", (u_price, u_name))
                        if cursor.rowcount > 0:
                            conn.commit()
                            st.success(f"✅ Updated price for '{u_name}'")
                        else:
                            st.warning("❌ Game not found")
                    except Exception as e:
                        st.error(f"Error: {e}")

        with tab3:
            with st.form("del_game"):
                st.subheader("Delete Game")
                d_name = st.text_input("Game Name to Delete")
                submitted = st.form_submit_button("Delete", type="primary")
                if submitted and d_name:
                    try:
                        cursor.execute("DELETE FROM Game WHERE Name = ?", (d_name,))
                        if cursor.rowcount > 0:
                            conn.commit()
                            st.success(f"✅ Deleted '{d_name}'")
                        else:
                            st.warning("❌ Game not found")
                    except Exception as e:
                        st.error(f"Error: {e}")

if __name__ == "__main__":
    main()

Overwriting app.py


In [17]:
import os
import time
import subprocess
from pyngrok import ngrok

def main():
    # Clean up
    os.system("pkill ngrok")
    os.system("pkill -f streamlit")
    time.sleep(2)

    try:
        # Start Streamlit - explicitly specify all parameters
        cmd = [
            "streamlit", "run", "app.py",
            "--server.port=8501",
            "--server.address=0.0.0.0",
            "--browser.serverAddress=0.0.0.0",
            "--browser.gatherUsageStats=false",
            "--server.headless=true"
        ]

        print("🚀 Starting Streamlit...")
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        time.sleep(10)  # Wait for Streamlit to fully start

        # Start ngrok
        print("🌐 Starting ngrok...")
        ngrok.set_auth_token("35rrDx80CY0SgLJ3c9urIpShXb4_6XM1EEFdNb41AJqn2YrVn")
        tunnel = ngrok.connect(8501, "http")
        public_url = tunnel.public_url

        print(f"🎉 YOUR APP IS AVAILABLE AT: {public_url}")
        print("If you see inspect page, wait 10 seconds and refresh")
        print("Press 0 to exit...")

        # Keep running until user inputs 0
        while True:
            user_input = input()
            if user_input == "0":
                print("Shutting down...")
                break
            time.sleep(1)

    except Exception as e:
        print(f"❌ Error: {e}")

    finally:
        # Cleanup
        os.system("pkill ngrok")
        os.system("pkill -f streamlit")
        print("Cleanup completed.")

if __name__ == "__main__":
    main()

🚀 Starting Streamlit...
🌐 Starting ngrok...
🎉 YOUR APP IS AVAILABLE AT: https://hibernal-unretorted-tresa.ngrok-free.dev
If you see inspect page, wait 10 seconds and refresh
Press 0 to exit...
0
Shutting down...
Cleanup completed.
